# PhysNet training on MD17 / rMD17 data

Train PhysNetJax on a **bring-your-own** MD17 or revised MD17 (rMD17) NPZ file.

**Input keys (rMD17):** `nuclear_charges`, `coords`, `energies`, `forces`  
**Units:** Å, kcal/mol, kcal/mol/Å

> rMD17 frames are correlated (MD trajectory). Use **≤ 1000 training structures**, not the full 100k set.

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# --- point this at your NPZ file ---
DATA_PATH = Path("/scicore/home/meuwly/boitti0000/data/rmd17/npz_data/rmd17_aspirin.npz")

# rMD17 recommendation: <= 1000 train frames
N_TRAIN = 950
N_VALID = 50
SEED = 42

NUM_EPOCHS = 50
BATCH_SIZE = 16
LEARNING_RATE = 1e-3

MAX_STRUCTURES = None  # set e.g. 200 for a quick smoke test
CONVERT_TO_EV = True   # kcal/mol -> eV (matches PhysNetJax defaults)

CKPT_DIR = Path("checkpoints/rmd17_aspirin")
RUN_NAME = "rmd17_aspirin"

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.9")

## 2. Load rMD17 data

In [ ]:
import numpy as np
import jax

from mmml.data.rmd17 import load_rmd17_npz
from mmml.data.loaders import get_data_statistics

raw = np.load(DATA_PATH, allow_pickle=True)
print("NPZ keys:", list(raw.keys()))
for k in raw.keys():
    v = raw[k]
    print(f"  {k:20s} shape={getattr(v, 'shape', '?')}")

data = load_rmd17_npz(
    DATA_PATH,
    convert_to_ev=CONVERT_TO_EV,
    max_structures=MAX_STRUCTURES,
)

num_atoms = int(data["N"].max())
print(f"\nMolecule size: {num_atoms} atoms")
print(f"Loaded {len(data['E'])} structures")
print(get_data_statistics(data))

## 3. Train / validation split

In [ ]:
n_total = len(data["E"])
assert N_TRAIN + N_VALID <= n_total, (
    f"Need {N_TRAIN + N_VALID} structures, only {n_total} available"
)

rng = np.random.RandomState(SEED)
perm = rng.permutation(n_total)
train_idx = perm[:N_TRAIN]
valid_idx = perm[N_TRAIN : N_TRAIN + N_VALID]


def subset_arrays(d, idx):
    out = {}
    for key, value in d.items():
        if isinstance(value, np.ndarray) and value.shape[0] == n_total:
            out[key] = value[idx]
        else:
            out[key] = value
    return out


train_data = subset_arrays(data, train_idx)
valid_data = subset_arrays(data, valid_idx)

print(f"Train: {len(train_data['E'])}  Valid: {len(valid_data['E'])}")

## 4. Model & training

In [ ]:
from mmml.physnetjax.physnetjax.models.model import EF
from mmml.physnetjax.physnetjax.training.training import train_model

print("JAX devices:", jax.devices())

key = jax.random.PRNGKey(SEED)

model = EF(
    features=64,
    max_degree=0,
    num_iterations=3,
    num_basis_functions=32,
    cutoff=5.0,
    natoms=num_atoms,
    max_atomic_number=int(data["Z"].max()),
    charges=False,
    zbl=False,
)

ema_params, best_loss = train_model(
    key=key,
    model=model,
    train_data=train_data,
    valid_data=valid_data,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    num_atoms=num_atoms,
    energy_weight=1.0,
    forces_weight=52.91,
    data_keys=("R", "Z", "F", "E", "N"),
    name=RUN_NAME,
    ckpt_dir=CKPT_DIR,
    best=True,
    log_tb=False,
    print_freq=1,
)

print(f"\nDone. Best validation loss: {best_loss:.6f}")
print(f"Checkpoints: {CKPT_DIR}")

## 5. (Optional) Save converted MMML NPZ

If you prefer the CLI trainer later:
```bash
python examples/other/co2/physnet_train/trainer.py \
  --train aspirin_train.npz --valid aspirin_valid.npz \
  --natoms 21 --epochs 500 --energy-unit eV
```

In [ ]:
SAVE_CONVERTED = False

if SAVE_CONVERTED:
    out_dir = Path("rmd17_converted")
    out_dir.mkdir(exist_ok=True)
    np.savez(out_dir / "train.npz", **train_data)
    np.savez(out_dir / "valid.npz", **valid_data)
    print("Saved:", out_dir / "train.npz", out_dir / "valid.npz")